In [29]:
#This script loads and verifies the test dataset for the GeoGuessr project
import pandas as pd
from pathlib import Path
image_dir = Path("datasets/OpenWorld/extracted_test")

#Verifies the shape
csv_path = "datasets/OpenWorld/test.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())

#This confirms data loaded as expected
print("Metadata rows:", len(df))
print("Metadata columns:", len(df.columns))

#nunique is num of unique elements
print("Unique countries:", df["country"].nunique())

print("Country counts:")
print(df["country"].value_counts().head())

image_files = [
    p for p in image_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in {
        ".jpg"
    }
]

print("Extracted image files:", len(image_files))
print("Example image paths:", image_files[:5])


Dataset shape: (210122, 26)
                 id   latitude  longitude  \
0   547473234108938 -16.336027  45.628280   
1   826109781317024  50.855687  56.147997   
2  1006398440000844  37.956651  14.954485   
3  2943891539215481  12.373333  -8.909906   
4   122945119799579   7.510295  99.061884   

                                  thumb_original_url country  \
0  https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...      MG   
1  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      KZ   
2  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      IT   
3  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      ML   
4  https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...      TH   

                 sequence    captured_at  lon_bin  lat_bin      cell  ...  \
0  Z2sJMyDVAtHBu7UFGvnYXg  1660909818797       61       28  (61, 28)  ...   
1  6cTpNBSYQH6SeDsfz_0mEQ  1548919530404       64       78  (64, 78)  ...   
2  xhcEpYg3dKnzts5ejIiSBJ  1658912403000       53       68  (53, 68)  ...   
3  j6nwj

In [30]:
from pathlib import Path


image_dir = Path("datasets/OpenWorld/extracted_test")

# Find images inside extracted_test and all its subfolders
image_lookup = {
    p.stem: str(p)
    for p in image_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg"}
}

# Match image IDs to the CSV
df["id"] = df["id"].astype(str).str.strip()
df["image_path"] = df["id"].map(image_lookup)

print("Unique image IDs:", len(image_lookup))
print("Unique CSV IDs:", df["id"].nunique())
print("Matching IDs:", df["image_path"].notna().sum())
print("CSV IDs missing images:", df["image_path"].isna().sum())

print("\nExample matches:")
print(df[["id", "image_path", "country"]].head())

Unique image IDs: 210122
Unique CSV IDs: 210122
Matching IDs: 210122
CSV IDs missing images: 0

Example matches:
                 id                                         image_path country
0   547473234108938  datasets\OpenWorld\extracted_test\03\547473234...      MG
1   826109781317024  datasets\OpenWorld\extracted_test\01\826109781...      KZ
2  1006398440000844  datasets\OpenWorld\extracted_test\00\100639844...      IT
3  2943891539215481  datasets\OpenWorld\extracted_test\00\294389153...      ML
4   122945119799579  datasets\OpenWorld\extracted_test\00\122945119...      TH


In [31]:
baseline_country = "US" #this was the highest occuring country in dataset

data = df["country"]
y_pred = [baseline_country] * len(data)

accuracy = (data == y_pred).mean()

print("Baseline country:", baseline_country)
print("Baseline accuracy:", accuracy)
print("Number of images:", len(data))
print("Number of correct guesses:", len(data) * accuracy)
print("Number of countries:", data.nunique())


Baseline country: US
Baseline accuracy: 0.11362446578654306
Number of images: 210122
Number of correct guesses: 23875.0
Number of countries: 219


In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image


# Define the image transformation
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_image(image_path):
    image = Image.open(image_path).convert("RGB")
    return transform(image).unsqueeze(0)  # Add batch dimension

def predict_image(image_path):
    image_tensor = load_image(image_path)
    with torch.no_grad():
        outputs = resnet(image_tensor)
    _, predicted = outputs.max(1)
    return predicted.item()

print(df[["id", "image_path", "unique_country"]])

import torch
import torch.nn as nn
from torchvision import models

weights = models.ResNet101_Weights.DEFAULT

resnet = models.resnet101(weights=weights)

# Replace the ImageNet classification layer
num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, df["country"].nunique())

print(resnet)




                      id                                         image_path  \
0        547473234108938  datasets\OpenWorld\extracted_test\03\547473234...   
1        826109781317024  datasets\OpenWorld\extracted_test\01\826109781...   
2       1006398440000844  datasets\OpenWorld\extracted_test\00\100639844...   
3       2943891539215481  datasets\OpenWorld\extracted_test\00\294389153...   
4        122945119799579  datasets\OpenWorld\extracted_test\00\122945119...   
...                  ...                                                ...   
210117   311190247232610  datasets\OpenWorld\extracted_test\03\311190247...   
210118   333786891410519  datasets\OpenWorld\extracted_test\02\333786891...   
210119   947201369432368  datasets\OpenWorld\extracted_test\00\947201369...   
210120  3975075185917527  datasets\OpenWorld\extracted_test\04\397507518...   
210121  4455441161137252  datasets\OpenWorld\extracted_test\03\445544116...   

       unique_country  
0                  MG  
1  

In [42]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["country_label"] = label_encoder.fit_transform(df["unique_country"])

num_countries = len(label_encoder.classes_)

print("Number of countries:", num_countries)
print("Country classes:", label_encoder.classes_)

Number of countries: 219
Country classes: ['AE' 'AF' 'AG' 'AI' 'AL' 'AM' 'AO' 'AR' 'AT' 'AU' 'AW' 'AX' 'AZ' 'BA'
 'BB' 'BD' 'BE' 'BF' 'BG' 'BH' 'BI' 'BJ' 'BM' 'BN' 'BO' 'BQ' 'BR' 'BS'
 'BT' 'BW' 'BY' 'BZ' 'CA' 'CD' 'CF' 'CG' 'CH' 'CI' 'CL' 'CM' 'CN' 'CO'
 'CR' 'CU' 'CV' 'CW' 'CX' 'CY' 'CZ' 'DE' 'DJ' 'DK' 'DM' 'DO' 'DZ' 'EC'
 'EE' 'EG' 'ER' 'ES' 'ET' 'FI' 'FJ' 'FK' 'FM' 'FO' 'FR' 'GA' 'GB' 'GD'
 'GE' 'GF' 'GG' 'GH' 'GL' 'GM' 'GN' 'GP' 'GQ' 'GR' 'GT' 'GW' 'GY' 'HK'
 'HN' 'HR' 'HT' 'HU' 'ID' 'IE' 'IL' 'IM' 'IN' 'IQ' 'IR' 'IS' 'IT' 'JE'
 'JM' 'JO' 'JP' 'KE' 'KG' 'KH' 'KI' 'KM' 'KN' 'KP' 'KR' 'KW' 'KY' 'KZ'
 'LA' 'LB' 'LC' 'LK' 'LR' 'LS' 'LT' 'LU' 'LV' 'LY' 'MA' 'MD' 'ME' 'MF'
 'MG' 'MK' 'ML' 'MM' 'MN' 'MQ' 'MR' 'MS' 'MT' 'MU' 'MV' 'MW' 'MX' 'MY'
 'MZ' 'NC' 'NE' 'NG' 'NI' 'NL' 'NO' 'NP' 'NZ' 'OM' 'PA' 'PE' 'PF' 'PG'
 'PH' 'PK' 'PL' 'PR' 'PS' 'PT' 'PW' 'PY' 'QA' 'RE' 'RO' 'RS' 'RU' 'RW'
 'SA' 'SB' 'SD' 'SE' 'SG' 'SI' 'SJ' 'SK' 'SL' 'SN' 'SO' 'SR' 'SS' 'ST'
 'SV' 'SX' 'SY' 'SZ' 'TC' 'TD' 'TG'

In [60]:
# Load and preprocess an image
image_tensor = load_image(
    "datasets/OpenWorld/extracted_test/00/100609922892497.jpg"
)

# Predict the country
resnet.eval()

with torch.no_grad():
    outputs = resnet(image_tensor)

predicted_class = outputs.argmax(dim=1).item()

# Convert the predicted index into a country name
predicted_country = label_encoder.inverse_transform([predicted_class])[0]

print("Predicted class index:", predicted_class)
print("Predicted country:", predicted_country)

Predicted class index: 31
Predicted country: BZ
